# Day 5 — ILT 1: Building the Bronze Layer — Design & Strategy
### GlobalMart Data Engineering · 9:30 AM – 11:00 AM

---

## Session Objectives

By the end of this session you will be able to:
- Explain the role and responsibilities of the Bronze layer in medallion architecture
- Apply the 3 Golden Rules of Bronze design
- Define the standard audit columns every Bronze table must have
- Design Bronze table schemas for all 4 GlobalMart sources
- Choose the correct write mode (append vs overwrite) per source
- Describe how CDC events are stored in Bronze

---

## Agenda

| Time | Topic |
|------|-------|
| 9:30 | Bronze in the medallion architecture |
| 9:40 | The 3 Golden Rules of Bronze |
| 9:50 | Audit columns — what, why, which |
| 10:05 | Table naming conventions |
| 10:15 | Write modes — append vs overwrite |
| 10:25 | CDC Bronze design |
| 10:40 | GlobalMart Bronze table designs |
| 10:55 | What NOT to do in Bronze + Q&A |

---
## 1. Bronze in the Medallion Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                  MEDALLION ARCHITECTURE                       │
│                                                               │
│  Sources → BRONZE → SILVER → GOLD → Consumers                │
│                                                               │
│  BRONZE   = Raw ingestion layer                               │
│             "Land first, ask questions later"                 │
│                                                               │
│  SILVER   = Cleaned, conformed, enriched                      │
│             "Make it trustworthy"                             │
│                                                               │
│  GOLD     = Business-ready aggregates                         │
│             "Make it useful"                                  │
└──────────────────────────────────────────────────────────────┘
```

### Bronze Is NOT a Staging Area

Common misconception: Bronze = temporary holding area, delete after Silver is built.

**Wrong.** Bronze is permanent.

| Staging Area | Bronze Layer |
|-------------|-------------|
| Temporary — deleted after processing | Permanent — kept forever |
| No audit trail | Full audit trail (when + from where) |
| No time travel | Delta time travel across all historical loads |
| No schema enforcement | Evolves with the source, tracked by Auto Loader |
| Can't reprocess | Can always reprocess Silver from Bronze |

### Why Keep Bronze Forever?

> **Bronze is your insurance policy.**
>
> If Silver has a bug (wrong transformation, bad join), you don't go back to the source.
> You reprocess from Bronze — faster, cheaper, and the source may have already changed.

Real example:
```
Silver job has a timezone bug → orders from 11 PM to 1 AM assigned wrong date
Fix: correct the transformation, rerun Silver from Bronze → problem solved
Without Bronze: would need to re-extract from Postgres, API, Neo4j, ADLS — painful
```

---
## 2. The 3 Golden Rules of Bronze

### Rule 1 — Land Raw, Transform Never

```
Bronze DOES:                          Bronze DOES NOT:
  ✅ Append incoming data               ❌ Rename columns
  ✅ Add audit columns                  ❌ Cast data types
  ✅ Store CDC events as-is             ❌ Join with other tables
  ✅ Preserve source column names       ❌ Filter rows
  ✅ Handle schema evolution            ❌ Aggregate
  ✅ Write to Delta                     ❌ Apply business rules
```

The only columns Bronze adds are **audit columns** — everything from the source lands as-is.

---

### Rule 2 — Idempotent Writes

**Idempotent** = running the ingestion multiple times produces the same result. No duplicates.

```
Run 1: orders.csv → 1,400,000 rows in Bronze
Run 2: orders.csv → same 1,400,000 rows (not re-processed)
Run 3: orders_new.csv → 1,400,005 rows (only 5 new rows added)
```

How we achieve this per source:
- **ADLS files:** Auto Loader checkpoint — tracks every processed file
- **Postgres CDC:** WAL offset — Lakeflow tracks last consumed event
- **APIs:** Watermark on `_ingested_date` — don't re-fetch already landed dates
- **Neo4j:** Full overwrite with timestamp — idempotent by design

---

### Rule 3 — Never Delete, Only Append (Except Snapshots)

```
Bronze append sources (Postgres, APIs, ADLS):
  ✅ NEVER delete rows from Bronze
  ✅ If source deletes a record → Bronze stores the DELETE event
  ✅ Silver handles MERGE to reflect the deletion

Bronze snapshot sources (Neo4j):
  ✅ Full overwrite is acceptable (no event log in graph DB)
  ✅ Use Delta time travel to access yesterday's snapshot
  ❌ Do NOT append snapshots — 365 full copies is expensive
```

---
## 3. Audit Columns — The Bronze Standard

Every Bronze table must have these audit columns added during ingestion:

| Column | Data Type | Value | Purpose |
|--------|-----------|-------|----------|
| `_ingested_at` | timestamp | `current_timestamp()` | When the row was written to Bronze |
| `_source_file` | string | `input_file_name()` | Which file this row came from (ADLS sources) |
| `_source_system` | string | `lit("supabase")` | Which source system produced this row |
| `_batch_id` | string | UUID or run ID | Which pipeline run produced this row |
| `_op` | string | INSERT / UPDATE / DELETE | CDC operation type (CDC sources only) |
| `_ts_ms` | long | Source event timestamp | When the event happened at source (CDC only) |

---

### Minimal vs Full Audit

```python
from pyspark.sql.functions import current_timestamp, input_file_name, lit
import uuid

batch_id = str(uuid.uuid4())   # unique per pipeline run

# Minimal (all sources):
df = df.withColumn("_ingested_at",   current_timestamp())
       .withColumn("_source_system", lit("adls_flat_files"))

# Full (file sources):
df = df.withColumn("_ingested_at",   current_timestamp())
       .withColumn("_source_file",   input_file_name())
       .withColumn("_source_system", lit("adls_flat_files"))
       .withColumn("_batch_id",      lit(batch_id))

# CDC sources — _op and _ts_ms come from the CDC event itself
# Lakeflow Connect adds them automatically
```

### Why `_batch_id`?

```
Scenario: Bronze has 1.4M rows from orders.csv
Bug: encoding issue corrupted 50,000 rows in batch_id = "abc-123"

Fix: DELETE FROM bronze WHERE _batch_id = 'abc-123'  ← surgical fix
     Re-run the ingestion job → rows re-land with new batch_id

Without _batch_id: would need to identify corrupted rows by content → painful
```

---
## 4. Table Naming Conventions

### Standard Pattern

```
bronze_{source_system}_{entity}

Examples:
  bronze_supabase_orders
  bronze_supabase_customers
  bronze_supabase_payments
  bronze_api_promotions
  bronze_api_exchange_rates
  bronze_neo4j_supplier_network
  bronze_adls_orders
  bronze_adls_products
  bronze_adls_shipping_tiers
```

### Path Convention in ADLS

```
abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/
  bronze/
    supabase/
      orders/           ← Delta table files
      customers/
      payments/
    api/
      promotions/
      exchange_rates/
    neo4j/
      supplier_network/
    adls/
      orders/
      products/
      shipping_tiers/
```

### Unity Catalog Naming (Week 3)

```sql
-- Full qualified name in Unity Catalog:
globalmart.bronze.supabase_orders
globalmart.bronze.api_promotions
globalmart.silver.orders_clean
globalmart.gold.fact_orders

-- Catalog = globalmart
-- Schema  = bronze / silver / gold
-- Table   = {source}_{entity}
```

---
## 5. Write Modes — Append vs Overwrite

### When to Append

Use **append** when the source produces an ordered stream of events:

```
Supabase CDC:      INSERT/UPDATE/DELETE events → append each event
ADLS Auto Loader:  new files land → append rows from each file
APIs:              daily fetch → append today's rows
```

```python
# Batch append:
df.write.format("delta").mode("append").save(bronze_path)

# Streaming append:
raw_stream.writeStream.format("delta").outputMode("append").start(bronze_path)
```

### When to Overwrite

Use **overwrite** when the source produces a full state snapshot (no event log):

```
Neo4j:      daily query returns FULL current graph → overwrite
Some APIs:  returns complete list (not just changes) → overwrite
```

```python
# Full overwrite:
df.write.format("delta").mode("overwrite").save(bronze_graph_path)
```

### The Danger of Overwrite on Append Sources

```
❌ Wrong: overwriting Bronze orders every day
   Day 1: 1,400,000 rows → Bronze has 1,400,000 rows
   Day 2: 1,500,000 rows → Bronze overwritten → 1,500,000 rows
   Day 3: discovers Day 1 data had a bug → CANNOT go back

✅ Correct: appending every day
   Day 1: 1,400,000 rows appended
   Day 2: 100,000 new rows appended
   Day 3: use time travel to go back to Day 1 state
```

| Source | Write Mode | Reason |
|--------|------------|--------|
| Supabase (CDC) | Append | Each CDC event is a new record |
| APIs | Append | Daily fetch adds new day's data |
| ADLS files | Append | Each file = new batch of rows |
| Neo4j | Overwrite | Full graph snapshot each run |

---
## 6. CDC Bronze Design

CDC (Change Data Capture) produces an **event log** — not the current state of the table.

### What CDC Events Look Like in Bronze

```
bronze_supabase_orders (CDC event log):

_op      | _ts_ms        | OrderID    | CustomerID | OrderDate  | ...
---------|---------------|------------|------------|------------
INSERT   | 1718438400000 | ORD-001    | CUST-001   | 2026-06-15 |
INSERT   | 1718438500000 | ORD-002    | CUST-002   | 2026-06-15 |
UPDATE   | 1718438600000 | ORD-001    | CUST-001   | 2026-06-15 |  ← status changed
INSERT   | 1718438700000 | ORD-003    | CUST-003   | 2026-06-15 |
DELETE   | 1718438800000 | ORD-002    | CUST-002   | 2026-06-15 |  ← cancelled
```

### Key Point: Bronze Stores ALL Events

```
Bronze does NOT apply MERGE — it appends EVERY event.

For ORD-001:
  Bronze has 2 rows: INSERT + UPDATE
  (Both rows exist in Bronze)

Silver applies MERGE:
  Takes the LATEST event per OrderID → keeps only current state
  silver_orders has 1 row for ORD-001 (the UPDATEd version)
```

### Why Store All Events in Bronze?

1. **Audit trail:** You can see every change ever made to any order
2. **Replayability:** Silver can be rebuilt with ANY version of the transformation logic
3. **Debugging:** If Silver shows wrong data, trace back to the exact CDC event
4. **Compliance:** Financial regulations often require full event logs

### The `_op` Column Values

| `_op` | Meaning | Silver Action |
|-------|---------|---------------|
| `r` or `INSERT` | Initial snapshot or new row | INSERT into Silver |
| `u` or `UPDATE` | Row was modified | MERGE (update) into Silver |
| `d` or `DELETE` | Row was deleted | MERGE (delete) or soft-delete flag in Silver |

---
## 7. GlobalMart Bronze Table Designs

### Source 1: Supabase Orders (CDC)

```
bronze_supabase_orders:
  _op              STRING    ← CDC operation (INSERT/UPDATE/DELETE)
  _ts_ms           LONG      ← CDC event timestamp (milliseconds)
  _ingested_at     TIMESTAMP ← when Bronze wrote this row
  _source_system   STRING    ← 'supabase'
  OrderID          STRING
  CustomerID       STRING
  OrderDate        STRING    ← kept as STRING in Bronze (Silver casts to DATE)
  ShippingDate     STRING
  OrderChannel     STRING
  ... (all source columns)
```

### Source 2: API Promotions (Batch)

```
bronze_api_promotions:
  _ingested_at     TIMESTAMP
  _source_system   STRING    ← 'promotions_api'
  _batch_id        STRING    ← which API call fetched this
  promo_id         STRING
  discount_pct     DOUBLE
  start_date       STRING
  end_date         STRING
  applicable_sku   ARRAY<STRING>
  ... (all API response fields)
```

### Source 3: Neo4j Supplier Network (Snapshot Overwrite)

```
bronze_neo4j_supplier_network:
  _ingested_at     TIMESTAMP
  _source_system   STRING    ← 'neo4j'
  SupplierID       STRING
  SupplierName     STRING
  ProductID        STRING
  Category         STRING
  relationship     STRING    ← SUPPLIES / SHIPS_TO / etc.
  ... (flattened from Cypher query)
```

### Source 4: ADLS Flat Files (Auto Loader)

```
bronze_adls_orders:
  _ingested_at     TIMESTAMP
  _source_file     STRING    ← full ADLS path of source CSV
  _source_system   STRING    ← 'adls'
  OrderID          STRING
  CustomerID       STRING
  OrderDate        STRING
  ShippingTierID   STRING
  OrderChannel     STRING
  ... (all CSV columns)
```

In [ ]:
# ─── ILLUSTRATIVE: Standard Bronze write pattern ──────────────────────────────
# Reference snippet showing the audit column pattern for a batch source.

"""
from pyspark.sql.functions import current_timestamp, lit
import uuid

# Replace with real connection details
df_raw = spark.read.format("jdbc") \
    .option("url",      "jdbc:postgresql://host/db") \
    .option("dbtable",  "orders") \
    .option("user",     "user") \
    .option("password", "YOUR_SUPABASE_PASSWORD") \
    .load()

batch_id = str(uuid.uuid4())

df_bronze = (
    df_raw
    .withColumn("_ingested_at",   current_timestamp())
    .withColumn("_source_system", lit("supabase"))
    .withColumn("_batch_id",      lit(batch_id))
)

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/supabase/orders")

print(f"Bronze write complete. Rows: {df_bronze.count():,} | Batch: {batch_id}")
"""

print("Key patterns in every Bronze write:")
print("  1. withColumn('_ingested_at', current_timestamp())")
print("  2. withColumn('_source_system', lit('source_name'))")
print("  3. withColumn('_batch_id', lit(batch_id))")
print("  4. .mode('append') for event sources")
print("  5. .option('mergeSchema', 'true') for schema evolution")

---
## 8. What NOT to Do in Bronze

| Anti-Pattern | Why It's Wrong | Correct Approach |
|-------------|---------------|------------------|
| Casting `OrderDate` STRING → DATE | If casting fails, you lose data | Cast in Silver |
| Renaming `OrderChannel` to `channel` | Breaks traceability to source | Rename in Silver |
| Filtering cancelled orders | Might need them later for analysis | Filter in Silver |
| Joining orders to customers | Bronze tables are independent | Join in Silver |
| Aggregating daily totals | Aggregation belongs in Gold | Aggregate in Gold |
| Deduplicating | Dedup logic may change | Dedup in Silver |
| Dropping columns that seem unused | Nothing is unused until proved otherwise | Keep all columns |

---

## Key Takeaways

1. **Bronze = permanent raw layer** — not a staging area, kept forever
2. **3 Golden Rules:** land raw, idempotent writes, never delete (except snapshots)
3. **Audit columns:** `_ingested_at`, `_source_file`, `_source_system`, `_batch_id` on every table
4. **CDC Bronze = event log** — append ALL events, Silver does the MERGE
5. **Write mode:** append for event sources, overwrite for snapshots (Neo4j)
6. **Never transform in Bronze** — land first, transform in Silver

---

## Discussion Questions

1. *A Supabase DELETE event arrives. Should Bronze delete the row from the table? Why or why not?*

2. *Your Bronze table has 200 million rows. You discover a bug in the ingestion job that corrupted rows from last Tuesday. How does `_batch_id` help you fix this surgically?*

3. *A junior engineer suggests casting all date columns to `DATE` type in Bronze to make Silver simpler. What's the risk?*

4. *Why does Neo4j Bronze use overwrite mode while Supabase Bronze uses append mode?*

5. *What happens to Bronze if the source schema adds a new column mid-stream? What config prevents a pipeline crash?*